# Phase 5 - Variational Quantum Classifier: Amplitude Encoding

A **variational quantum classifier (VQC)** has three parts:

1. A **feature map** that encodes the data into a quantum state (this is the
   part that changes between notebooks - here it is **amplitude encoding**).
2. A trainable **ansatz**, a circuit with adjustable rotation parameters.
3. A classical **optimizer** that tunes those parameters to reduce
   classification error.

Amplitude encoding stores the feature vector directly in the state amplitudes. The data must be padded to a power-of-two length and normalized, so we use the `pad_and_normalize` helper. Because the state-preparation step cannot provide an analytic gradient, we again use the gradient-free COBYLA optimizer.

We reuse the exact dataset, split, and scaling from Phase 2 so the result is
directly comparable to the classical baseline and to the other encodings.

In [ ]:
"""05_vqc_amplitude_encoding.ipynb"""

# Cell 01 - Imports and reproducible seed

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import display
from qiskit.circuit.library import real_amplitudes
from qiskit_machine_learning.algorithms.classifiers import VQC
from qiskit_machine_learning.circuit.library import raw_feature_vector
from qiskit_machine_learning.optimizers import COBYLA
from qiskit_machine_learning.utils import algorithm_globals

import qml_utils as hu

algorithm_globals.random_seed = hu.RANDOM_SEED
np.random.seed(hu.RANDOM_SEED)

---
**Cell 02 - Load, scale, then pad and normalize.**
Amplitude encoding needs an extra preparation step. We first scale the data
(here into $[0, 1]$ so all amplitudes are non-negative), then pad each
2-feature sample to 4 values and normalize to unit length. The result has 4
amplitudes, which fit on $\log_2 4 = 2$ qubits.

In [ ]:
# Cell 02 - Prepare data for amplitude encoding

x_train, x_test, y_train, y_test = hu.load_moons(n_samples=200, noise=0.20)

# Scale to [0, 1] so the padded amplitudes are non-negative, then normalize
x_train_scaled, x_test_scaled, scaler = hu.scale_features(
    x_train, x_test, feature_range=(0.0, 1.0)
)
x_train_amp = hu.pad_and_normalize(x_train_scaled, num_amplitudes=4)
x_test_amp = hu.pad_and_normalize(x_test_scaled, num_amplitudes=4)

print(f"Prepared training shape: {x_train_amp.shape}")
print(f"Example normalized row : {x_train_amp[0]}")
print(f"Its norm (should be 1) : {np.linalg.norm(x_train_amp[0]):.6f}")

---
**Cell 03 - Build the encoding and the ansatz.**
The feature map is `raw_feature_vector`, which loads a 4-amplitude vector
onto 2 qubits. The ansatz is the same `real_amplitudes` circuit used in the
other VQC notebooks, so the only thing that changes between phases is the
encoding.

**About the drawing.** Amplitude encoding does not correspond to a fixed,
hand-written sequence of gates. Instead Qiskit uses an `initialize`
instruction that, once the amplitudes are known, *synthesizes* a circuit to
build that exact state. If you only call `.decompose()` you get a single
opaque box labeled `isometry_to_uncompute_dg`: that is an internal step of the
synthesis algorithm (it finds gates that uncompute the target state back to
$|00
angle$, then daggers them), not the final gate list. To see the gates
that actually run, **transpile** the circuit to a concrete gate set. We show
both views below so the difference between the high-level abstraction and the
executable circuit is clear.

You will also see `reset` gates in the transpiled version. That is by design:
`initialize` means "reset the qubits to $|0
angle$, then prepare the state,"
so the resets are simply the first half of that definition.

In [ ]:
# Cell 03 - Feature map + ansatz

from qiskit import transpile

feature_map = raw_feature_vector(feature_dimension=4)
ansatz = real_amplitudes(num_qubits=2, reps=2)

print(f"Number of qubits     : {feature_map.num_qubits}")
print(f"Trainable parameters : {ansatz.num_parameters}")

# raw_feature_vector cannot be drawn with unbound parameters, so bind one
# example sample to visualize its state-preparation circuit.
example = feature_map.assign_parameters(x_train_amp[0])

# View 1: the high-level abstraction. .decompose() only unwraps one layer and
# exposes the synthesis algorithm's internal "isometry_to_uncompute" box.
print("High-level view (decompose) - shows the opaque isometry instruction:")
display(example.decompose().draw(output="mpl"))

# View 2: the real gates. Transpiling to a concrete basis synthesizes the
# state preparation into executable gates (note the reset + u + cx gates).
runnable = transpile(example, basis_gates=["u", "cx"], optimization_level=3)
print(
    f"Executable view (transpiled) - depth {runnable.depth()}, ops {dict(runnable.count_ops())}:"
)
display(runnable.draw(output="mpl"))

---
**Cell 04 - Train the classifier.**
Same training recipe as the other VQCs: COBYLA with a callback that records
the objective. The only difference is the prepared (padded, normalized) data.

In [ ]:
# Cell 04 - Train the amplitude-encoding VQC

objective_values = []


def store_objective(weights, value):
    objective_values.append(value)


vqc = VQC(
    feature_map=feature_map,
    ansatz=ansatz,
    optimizer=COBYLA(maxiter=80),
    callback=store_objective,
)

vqc.fit(x_train_amp, y_train)

amp_train_acc = vqc.score(x_train_amp, y_train)
amp_test_acc = vqc.score(x_test_amp, y_test)
print(f"Amplitude-encoding VQC training accuracy: {amp_train_acc:.3f}")
print(f"Amplitude-encoding VQC test accuracy    : {amp_test_acc:.3f}")

---
**Cell 05 - Training convergence.**

In [ ]:
# Cell 05 - Plot the optimization curve

plt.figure(figsize=(7, 4))
plt.plot(objective_values)
plt.xlabel("optimizer iteration")
plt.ylabel("objective value")
plt.title("Amplitude-encoding VQC training convergence")
plt.grid(True, alpha=0.3)
plt.show()

---
**Cell 06 - Decision boundary.**
Here the grid points must be prepared exactly like the training data before
the classifier can read them. We pass `pad_and_normalize` as the `transform`
so every grid point is padded and normalized on its way into `vqc.predict`.
We plot over the $[0, 1]$-scaled feature space used for this encoding.

In [ ]:
# Cell 06 - Decision boundary with the amplitude-preparation transform


def prepare_grid(points):
    return hu.pad_and_normalize(points, num_amplitudes=4)


hu.plot_decision_boundary(
    vqc.predict,
    x_test_scaled,
    y_test,
    title=f"Amplitude-encoding VQC (test accuracy {amp_test_acc:.2f})",
    grid_steps=40,
    transform=prepare_grid,
)
plt.show()

---
## Interpretation

Amplitude encoding is compact (only 2 qubits for 4 amplitudes) but the
normalization step discards the magnitude of each point, keeping only its
direction. Watch for how that limitation shows up in the decision boundary
and in the accuracy. Record the **test accuracy** for Phase 7.